##Loading saved weights in drive

In [1]:
!gdown --id 1Bit1nc1saxYfbPNZx4DyNvz2jQQ6JWft -O checkpoint-481.zip
# https://drive.google.com/file/d/1Bit1nc1saxYfbPNZx4DyNvz2jQQ6JWft/view?usp=sharing

/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Bit1nc1saxYfbPNZx4DyNvz2jQQ6JWft
From (redirected): https://drive.google.com/uc?id=1Bit1nc1saxYfbPNZx4DyNvz2jQQ6JWft&confirm=t&uuid=5920c1e5-8938-4a4a-a717-0b141332097f
To: /kaggle/working/checkpoint-481.zip
100%|██████████████████████████████████████| 44.4M/44.4M [00:00<00:00, 76.8MB/s]


In [15]:
!zip -r /kaggle/working/outputs/checkpoint-497.zip /kaggle/working/outputs/checkpoint-497


  adding: kaggle/working/outputs/checkpoint-497/ (stored 0%)
  adding: kaggle/working/outputs/checkpoint-497/tokenizer_config.json (deflated 94%)
  adding: kaggle/working/outputs/checkpoint-497/tokenizer.json (deflated 85%)
  adding: kaggle/working/outputs/checkpoint-497/training_args.bin (deflated 51%)
  adding: kaggle/working/outputs/checkpoint-497/adapter_config.json (deflated 56%)
  adding: kaggle/working/outputs/checkpoint-497/special_tokens_map.json (deflated 71%)
  adding: kaggle/working/outputs/checkpoint-497/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/outputs/checkpoint-497/README.md (deflated 66%)


In [16]:
from IPython.display import FileLink

# Display the link
display(FileLink(r'outputs/checkpoint-497.zip'))


/kaggle/working/outputs/checkpoint-497.zip

In [2]:
!unzip checkpoint-481.zip -d checkpoint-481

Archive:  checkpoint-481.zip
   creating: checkpoint-481/kaggle/working/outputs/checkpoint-481/
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/tokenizer_config.json  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/adapter_model.safetensors  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/special_tokens_map.json  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/tokenizer.json  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/README.md  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/training_args.bin  
  inflating: checkpoint-481/kaggle/working/outputs/checkpoint-481/adapter_config.json  


In [1]:
print("working")

working


#####FINETUNING SETUP STARTS HERE

In [2]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth
!pip install --upgrade --no-cache-dir transformers

In [3]:
# !pip install "trl<0.15.0"

In [4]:
from unsloth import FastLanguageModel
import torch
import pandas as pd
from datasets import Dataset  # Import Hugging Face Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from datasets import DatasetDict

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/working/outputs/checkpoint-495",#"/kaggle/working/outputs/checkpoint-39", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.3.5: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 6.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Unsloth 2025.3.5 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [6]:
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj",],
#     lora_alpha = 16,
#     lora_dropout = 0, # Supports any, but = 0 is optimized
#     bias = "none",    # Supports any, but = "none" is optimized
#     # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
#     random_state = 3407,
#     use_rslora = False,  # We support rank stabilized LoRA
#     loftq_config = None, # And LoftQ
# )

In [7]:
device = torch.device("cuda")
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [8]:


# Load your dataset
df = pd.read_csv('/kaggle/input/train-data/train_data.csv')

# Create the Llama_dataset DataFrame
Llama_dataset = pd.DataFrame(columns=['Question', 'Answer'])

# Split the formatted_recipe column into Question and Answer
Llama_dataset[['Question', 'Answer']] = df['formatted_recipe'].str.split('<INGR_START>', expand=True)

# Add '<INGR_START>' to the beginning of each Answer
word_to_add = '<INGR_START>'
Llama_dataset['Answer'] = Llama_dataset['Answer'].apply(lambda x: f"{word_to_add} {x}")



In [9]:


# Define batch size
batch_size = 4000  # Adjust this based on your available memory
num_batches = len(Llama_dataset) // batch_size + (1 if len(Llama_dataset) % batch_size != 0 else 0)

# Split the dataset into batches
batches = []
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min(start_idx + batch_size, len(Llama_dataset))
    batch = Llama_dataset.iloc[start_idx:end_idx]  # Use iloc for slicing
    batches.append(batch)

In [10]:
batches[0]

,Question,Answer
0,<RECIPE_START> <INPUT_START> potatoes <NEXT_IN...,<INGR_START> 2 cans (14 ounces each) sauerkra...
1,<RECIPE_START> <INPUT_START> potatoes <NEXT_IN...,<INGR_START> 3 medium potatoes <NEXT_INGR> 1 ...
2,<RECIPE_START> <INPUT_START> lemon wedges <NEX...,<INGR_START> 3 tablespoons pure olive oil <NE...
3,<RECIPE_START> <INPUT_START> red wine vinegar ...,<INGR_START> 1 lb. ground beef <NEXT_INGR> 1/...
4,<RECIPE_START> <INPUT_START> oregano <NEXT_INP...,<INGR_START> 2 ripe tomatoes <NEXT_INGR> 1/2 ...
...,...,...
3995,<RECIPE_START> <INPUT_START> mozzarella cheese...,<INGR_START> 1 (6 ounce) box herb stuffing mi...
3996,<RECIPE_START> <INPUT_START> sugar <NEXT_INPUT...,<INGR_START> 1 1/2 c. sugar <NEXT_INGR> 3 Tbs...
3997,<RECIPE_START> <INPUT_START> drizzle olive oil...,<INGR_START> 3 cups peeled and diced pumpkin ...
3998,<RECIPE_START> <INPUT_START> margarine <NEXT_I...,<INGR_START> 1/2 c white Karo syrup <NEXT_ING...


In [11]:
from datasets import Dataset
from unsloth import standardize_sharegpt

def process_dataframe_for_chat_template(df, tokenizer, chat_template="llama-3.1"):
    """
    Processes a DataFrame into a formatted dataset suitable for training with a chat template.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'Question' and 'Answer' columns.
        tokenizer: The tokenizer to be used for applying the chat template.
        chat_template (str): The chat template to apply (default is "llama-3.1").

    Returns:
        Dataset: The processed dataset ready for training.
    """
    # Convert the DataFrame into the required format for the chat template
    conversations = []
    for _, row in df.iterrows():
        conversation = [
            {"role": "user", "content": row['Question']},
            {"role": "assistant", "content": row['Answer']},
        ]
        conversations.append(conversation)

    # Create a Hugging Face Dataset from the conversations
    dataset = Dataset.from_dict({"conversations": conversations})

    # Standardize the dataset using unsloth's standardize_sharegpt
    dataset = standardize_sharegpt(dataset)

    # Define the formatting function
    def formatting_prompts_func(examples):
        convos = examples["conversations"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
        return {"text": texts}

    # Apply the chat template to the tokenizer
    tokenizer = get_chat_template(tokenizer, chat_template=chat_template)

    # Apply the formatting function to the dataset
    dataset = dataset.map(formatting_prompts_func, batched=True)

    return dataset

In [12]:
num_batches=len(batches)
num_batches

497

In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
import os

# Function to train on batches and save checkpoints
def train_on_batches(batches, model, tokenizer, max_seq_length, output_dir="outputs",l=0):
    """
    Trains the model on batched data and saves checkpoints after each batch.

    Args:
        batches (list): List of DataFrames (batches).
        model: The pre-trained model to fine-tune.
        tokenizer: The tokenizer associated with the model.
        max_seq_length (int): Maximum sequence length for training.
        output_dir (str): Directory to save checkpoints.
    """
    for i in range(l,num_batches):
        print(f"Processing batch {i + 1}/{len(batches)}")
        batch=batches[i]
        # Convert the batch to ShareGPT format
        dataset = process_dataframe_for_chat_template(batch, tokenizer)

        # Initialize the SFTTrainer for the current batch
        trainer = SFTTrainer(
             model=model,
             tokenizer=tokenizer,
             train_dataset=dataset,
             dataset_text_field="text",
             max_seq_length=max_seq_length,
             data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
             dataset_num_proc=4,
             packing=False,  # Can make training 5x faster for short sequences
             args=TrainingArguments(
                 per_device_train_batch_size=4,
                 gradient_accumulation_steps=4,
                 warmup_steps=2,
                 num_train_epochs = 1,
                 #max_steps=60,  # Adjust based on your batch size and dataset
                 learning_rate=2e-5,
                 fp16=not is_bfloat16_supported(),
                 bf16=is_bfloat16_supported(),
                 logging_steps=1,
                 optim="adamw_8bit",
                 weight_decay=0.01,
                 lr_scheduler_type="linear",
                 seed=3407,
                 output_dir=output_dir,
                 save_steps=300,  # Save a checkpoint every 10 steps
                 save_total_limit=1,  # Keep only the last 2 checkpoints
                 report_to="none",  # Use this for WandB etc
             ),
         )
        

        # Apply train_on_responses_only to focus on assistant responses
        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
            response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
        )

        # Debugging: Inspect tokenized input and labels
        print("Sample input IDs:", tokenizer.decode(trainer.train_dataset[5]["input_ids"]))
        space = tokenizer(" ", add_special_tokens=False).input_ids[0]
        print("Sample labels:", tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]]))

        # Train on the current batch
        print(f"Training on batch {i + 1}/{len(batches)}")
        trainer.train()
        # Save the model checkpoint after each batch
        checkpoint_dir = os.path.join(output_dir, f"checkpoint-{i + 1}")
        trainer.save_model(checkpoint_dir)
        print(f"Model saved to {checkpoint_dir}")

In [14]:
train_on_batches(batches, model, tokenizer, max_seq_length, output_dir="outputs",l=495)

Processing batch 496/497


Standardizing format:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Tokenizing to ["text"] (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

...

Map (num_proc=4):   0%|          | 0/1233 [00:00<?, ? examples/s]

Sample input IDs: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

<RECIPE_START> <INPUT_START> oregano <NEXT_INPUT> ground cumin <NEXT_INPUT> ground turmeric <NEXT_INPUT> schug <NEXT_INPUT> lemon juice <NEXT_INPUT> kale <NEXT_INPUT> vegetable <NEXT_INPUT> walnuts <NEXT_INPUT> kosher salt <NEXT_INPUT> olive oil <INPUT_END> <|eot_id|><|start_header_id|>assistant<|end_header_id|>

<INGR_START>  1 tablespoon olive oil <NEXT_INGR> 1/2 teaspoon schug (or to taste) <NEXT_INGR> 1 teaspoon ground cumin <NEXT_INGR> 1 teaspoon ground turmeric <NEXT_INGR> 1 teaspoon dried oregano <NEXT_INGR> 2 cups tightly packed kale, rinsed and coarsely chopped <NEXT_INGR> 1 teaspoon kosher salt, divided <NEXT_INGR> 1/2 cup vegetable or vegetarian broth <NEXT_INGR> 1 tablespoon fresh lemon juice <NEXT_INGR> 1/2 cup coarsely chopped walnuts, toasted <INGR_END> <INSTR_START> In a large s

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,233 | Num Epochs = 1 | Total steps = 77
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192/785,713,152 (1.43% trained)


Step,Training Loss
1,0.982000
2,1.016100
3,1.019700
4,0.960600
5,1.057000
6,0.832200
7,1.083900
8,0.949200
9,1.033400
10,1.114200


Model saved to outputs/checkpoint-497


In [ ]:
test_data = pd.read_csv("/kaggle/input/recipe2m-test/test_dataset.csv")

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.gleu_score import sentence_gleu
from datasets import Dataset

def process_dataframe_with_split(df):
    messages = []
    
    for _, row in df.iterrows():
        messages.append({
            "role": "user",
            "content": row["Prompt"],
            "reference": row["Response"]  # Keeping "reference" if needed
        })
    
    return messages

test_dataset = process_dataframe_with_split(test_data.iloc[[0, 2, 4]])

input_dataset = [{'role': item['role'], 'content': item['content']} for item in test_dataset]

print(test_dataset[0])
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)
FastLanguageModel.for_inference(model)

inputs = tokenizer.apply_chat_template(
    input_dataset,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=300, use_cache=True, temperature=1.5, min_p=0.1)
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

bleu_scores = []
gleu_scores = []

references = [item["reference"] for item in test_dataset]  # Extract references

for pred, ref in zip(predictions, references):
    reference_tokens = [ref.split()]
    prediction_tokens = pred.split()
    
    bleu = sentence_bleu(reference_tokens, prediction_tokens)
    gleu = sentence_gleu(reference_tokens, prediction_tokens)
    
    bleu_scores.append(bleu)
    gleu_scores.append(gleu)

average_bleu = sum(bleu_scores) / len(bleu_scores)
average_gleu = sum(gleu_scores) / len(gleu_scores)

print(f"Average BLEU Score: {average_bleu:.4f}")
print(f"Average GLEU Score: {average_gleu:.4f}")
print(predictions[0])